# Combine the coarsened ERA5 data

Author: Brian Green (briangre@stanford.edu)

In [2]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import netCDF4 as nc
import cartopy.crs as ccrs
import matplotlib.ticker as mticker
import xarray as xr
from scipy import interpolate

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    tqdm.pandas()

%matplotlib inline

### Load in the data and combine

In [3]:
files = np.array(['Sep2010','Oct2010','Nov2010','Dec2010','Jan2011'])
start_times = np.array(['2010-09-01','2010-10-01','2010-11-01','2010-12-01','2011-01-01']).astype('datetime64')
nfiles = files.size

# Load in the first two files
file_path = str('temp_data/' + files[0] + '_temperature_coarse.nc')
dataset = nc.Dataset(file_path)
time = np.array(dataset.variables['time']).astype('timedelta64[h]')
time = start_times[0] + time
lat = np.array(dataset.variables['latitude']) # (deg. N)
lon = np.array(dataset.variables['longitude']) # (deg. E)
pressure = np.array(dataset.variables['pressure']) # (hPa)
temperature = np.array(dataset.variables['__xarray_dataarray_variable__']) # (K)
file_path = str('temp_data/' + files[0] + '_N2_coarse.nc')
dataset = nc.Dataset(file_path)
N2 = np.array(dataset.variables['__xarray_dataarray_variable__']) # (s**-2)

# Load in the rest and append
for i in np.arange(nfiles-1):
    
    file_path = str('temp_data/' + files[i+1] + '_temperature_coarse.nc')
    dataset = nc.Dataset(file_path)
    temp_time = np.array(dataset.variables['time']).astype('timedelta64[h]')
    temp_time = start_times[i+1] + temp_time
    temp_temperature = np.array(dataset.variables['__xarray_dataarray_variable__']) # (K)
    file_path = str('temp_data/' + files[i+1] + '_N2_coarse.nc')
    dataset = nc.Dataset(file_path)
    temp_N2 = np.array(dataset.variables['__xarray_dataarray_variable__']) # (s**-2)
    
    time = np.concatenate((time,temp_time))
    temperature = np.concatenate((temperature,temp_temperature), axis=0)
    N2 = np.concatenate((N2,temp_N2), axis=0)

[n_time,n_pressure,n_lat,n_lon] = temperature.shape

### Save the data

In [5]:
df = xr.DataArray(temperature, coords=[('time',time), ('pressure',pressure),
                                      ('latitude',lat), ('longitude',lon)])
df.to_netcdf('temperature_coarse.nc')
print(df)
print()
print()

df = xr.DataArray(N2, coords=[('time',time), ('pressure',pressure),
                                     ('latitude',lat), ('longitude',lon)])
df.to_netcdf('N2_coarse.nc')
print(df)

<xarray.DataArray (time: 3672, pressure: 7, latitude: 11, longitude: 36)>
array([[[[225.56147062, 222.30842809, 218.58537401, ..., 215.72566385,
          218.00223168, 223.93015473],
         [224.34388662, 223.66394122, 221.97941953, ..., 218.67352449,
          217.70398873, 222.111606  ],
         [220.32794516, 221.00582062, 220.82732413, ..., 219.89203628,
          215.63400621, 216.99703581],
         ...,
         [195.24980225, 196.7027157 , 197.22944857, ..., 192.30168565,
          192.93100847, 193.76860329],
         [195.32442023, 195.62049718, 195.95932108, ..., 193.42565874,
          194.49587937, 195.01887406],
         [194.18285039, 194.3223949 , 194.30967792, ..., 193.3137679 ,
          193.7373376 , 194.03106019]],

        [[222.62729388, 218.18692278, 214.98983482, ..., 215.00961426,
          218.11590454, 221.35355176],
         [222.25979901, 220.59066545, 219.0821964 , ..., 219.1283409 ,
          218.93909844, 221.13230011],
         [218.81155761, 219.46